# Functional demonstration: debris

Reduced model: 8 populations, 2 random seeds, and a limited parameter grid. These results are **not** the manuscript simulations. Run all cells in this notebook.


In [ ]:
#####| Fixed scenario condition    | What changes across seeds                                                                                |
#####| --------------------------- | -------------------------------------------------------------------------------------------------------- |
#####| `max_inter=0` and `L_var=0` | **No differences** (`inter=0`; `sigmas`/`lambdas` unchanged)                                             |
#####| `max_inter>0` and `L_var=0` | **Only** the interaction matrix `inter` differs                                                          |
#####| `max_inter=0` and `L_var>0` | **Only** `sigmas` / `lambdas` differ (and they affect **t ≥ 4h** since phase 1 uses `sigma=0, lambda=0`) |
#####| `max_inter>0` and `L_var>0` | Both `inter` and `sigmas` / `lambdas` differ                                                             |


# Mechanistic model for data generation under different scenarios
import numpy as np   #for numerical computing. It is used to handle arrays and matrix operations
from scipy.integrate import odeint  #for solving ordinary differential equations (ODEs)


# =============== define functions ===============
def generate_valid_random(N): ##Generate a random array of size N from a normal distribution such that its mean is less than or equal to 0.1 and its standard deviation is less than or equal to 1.2. If these conditions are not met, the array is regenerated
    while True:
        r = np.random.randn(N)  #randn is a function in NumPy that generates random numbers from a standard normal distribution (i.e., normal distribution with mean 0 and variance 1)
        if abs(np.mean(r)) <= 0.1 and np.std(r) <= 1.2:
            return r
            
def param_generator_lv(N, connectedness, neg_frac, max_neg, max_pos, rng): 
    # Generate an interaction matrix using RNG
    indsPerm = rng.permutation(N * (N - 1))
    inter_init = np.zeros((N, N - 1))
    
    neg_count = int(np.ceil(N * (N - 1) * connectedness * neg_frac))
    pos_count = int(np.floor(N * (N - 1) * connectedness * (1 - neg_frac)))
    
    inter_init.flat[indsPerm[:neg_count]] = -rng.random(neg_count) * max_neg
    inter_init.flat[indsPerm[neg_count:neg_count+pos_count]] = rng.random(pos_count) * max_pos
    
    inter = np.zeros((N, N)) 
    for i in range(N):
        mask = np.ones(N, dtype=bool)
        mask[i] = False
        inter[i, mask] = inter_init[i, :]
    
    return [inter, inter * (inter < 0), inter * (inter > 0)]


def calculate_growth_rates(x, params, max_Gs, sigma, lambda_val):
    # return G_i-L_i
    return (max_Gs * (1 - x) + params[0] @ x) * (1 - sigma) - lambda_val

def calculate_effective_growth_rates(x, params, max_Gs):
    # return G_i
    return (max_Gs * (1 - x) + params[0] @ x) 

def calculate_lysis_rates(x, params, max_Gs, sigma, lambda_val):
    # return L_i
    return (max_Gs * (1 - x) + params[0] @ x) * sigma + lambda_val


def core_ode_lv_residue(state, t, inter, max_Gs, sigma, lambda_val, li, de):

    S = state[:N]
    R = state[N:]

    # effective growth rate
    G = max_Gs * (1 - S) + inter @ S

    # phase switch
    if t < 4:
        sigma_eff = np.zeros_like(sigma)
        lambda_eff = np.zeros_like(lambda_val)
    else:
        sigma_eff = sigma
        lambda_eff = lambda_val

    # lysis rate
    L = sigma_eff * G + lambda_eff

    # dS/dt
    dSdt = S * (G - L)

    # dR/dt
    dRdt = li * L * S - de * R

    # enforce R >= 0
    mask = (R <= 0) & (dRdt < 0)
    dRdt[mask] = 0

    return np.concatenate([dSdt, dRdt])


def simulate_timecourse_with_rates(x0, tend, params, max_Gs, sigma, lambda_val, li, de):

    t = np.linspace(0, tend, 111)

    # initial state
    S0 = x0
    R0 = np.zeros(N)

    state0 = np.concatenate([S0, R0])

    solution = odeint(
        core_ode_lv_residue,
        state0,
        t,
        args=(params[0], max_Gs, sigma, lambda_val, li, de),
        atol=1e-9
    )

    # split solution
    y = solution[:, :N]
    R = solution[:, N:]

    growth_rates = np.zeros_like(y)
    effective_growth_rates = np.zeros_like(y)
    lysis_rates = np.zeros_like(y)
    Bi_residue = np.zeros_like(y)
    net_rate_resi = np.zeros_like(y)
    lysis_rate_with_resi = np.zeros_like(y)


    for i in range(len(t)):

        effective_growth_rates[i] = calculate_effective_growth_rates(y[i], params, max_Gs)
        
        if t[i] < 4:
            lysis_rates[i] = np.zeros_like(y[i])
        else:
            lysis_rates[i] = calculate_lysis_rates(y[i], params, max_Gs, sigma, lambda_val)

        growth_rates[i] = effective_growth_rates[i] - lysis_rates[i]
        # ===== residue quantities =====
        Bi_residue[i] = y[i] + R[i]
        numerator = y[i] * (effective_growth_rates[i] - lysis_rates[i] + li * lysis_rates[i]) - de * R[i]
        denom = np.maximum(Bi_residue[i], 1e-12)
        net_rate_resi[i] = numerator / denom
        lysis_rate_with_resi[i] = effective_growth_rates[i] - net_rate_resi[i]

    return t, y, growth_rates, effective_growth_rates, lysis_rates,R, Bi_residue, net_rate_resi, lysis_rate_with_resi


# ====================== Parameter settings ======================
N = 8
x0 = np.ones(N) / (100 * N)

tend = 11 #Simulation time (11 hours).
max_Gs = np.linspace(0.7, 1.5, N) #Maximum growth rates for each species, linearly spaced between 0.7 and 1.5.
sigma = 1.5 * np.ones(N)        # a
lambda_val = 0.1 * np.ones(N)   # b

connectedness = 1 #density of interactions between species (1 means fully connected).
neg_frac = 1
max_neg = 0
max_pos = max_neg * 0.5



# ====================== Initialize output arrays ====================== 
n_time = tend*10+1 # 10 time points per hour
n_species = 8
# parameter set
scenario_params = [{'L_var': lv, 'max_inter': mi} 
                  for lv in [0, 0.1] 
                  for mi in [1]]
n_scenarios = len(scenario_params) 

n_seeds = 2  # keep this to define the array size

# Initialize output arrays once for all seeds
outputs = {
    'population_data': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'growth_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'effective_growth_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'lysis_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'relative_abundance': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'residue': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'biomass_with_resid': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'net_rate_with_resid': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'lysis_rate_with_resid': np.zeros((n_time, n_species, n_seeds, n_scenarios))
}


for scenario_idx, scenario in enumerate(scenario_params):

    L_var = scenario['L_var']
    max_inter = scenario['max_inter']

    for seed in range(1, n_seeds + 1):
        np.random.seed(seed)  # set seed
        # RNG for interaction matrix (changes across seeds)
        rng = np.random.default_rng(seed)

        # impose lysis variability
        sigmas = sigma + L_var * generate_valid_random(N)
        lambdas = lambda_val + L_var * generate_valid_random(N)
        # residue parameters
        li = np.clip(np.random.normal(0,0.2,N),0,1)
        de = np.clip(np.random.normal(0, 0.1, N), 0, None)
                
        # generate interaction matrix
        current_max_inter = max_Gs[-1] * max_inter if max_inter > 0 else 0
        params = param_generator_lv(N, connectedness, neg_frac, 
                                    max_neg=current_max_inter, 
                                    max_pos=current_max_inter * 0.5, rng=rng)

        # simulate
        t, y, gr, egr, lr, resid, b_with, gr_with, lr_with = simulate_timecourse_with_rates(x0, tend, params, max_Gs, sigmas, lambdas, li, de)

        # store into outputs
        outputs['population_data'][:, :, seed-1, scenario_idx] = y
        outputs['growth_rates'][:, :, seed-1, scenario_idx] = gr
        outputs['effective_growth_rates'][:, :, seed-1, scenario_idx] = egr
        outputs['lysis_rates'][:, :, seed-1, scenario_idx] = lr
        outputs['relative_abundance'][:, :, seed-1, scenario_idx] = y / y.sum(axis=1, keepdims=True)
        outputs['residue'][:, :, seed-1, scenario_idx] = resid
        outputs['biomass_with_resid'][:, :, seed-1, scenario_idx] = b_with
        outputs['net_rate_with_resid'][:, :, seed-1, scenario_idx] = gr_with
        outputs['lysis_rate_with_resid'][:, :, seed-1, scenario_idx] = lr_with
        

# Save full dataset
np.savez(f'simulation_results_{n_seeds}seeds_with_debris.npz',
         time_points=t,
         population_data=outputs['population_data'],
         growth_rates=outputs['growth_rates'],
         effective_growth_rates=outputs['effective_growth_rates'],
         lysis_rates=outputs['lysis_rates'],
         relative_abundance=outputs['relative_abundance'],
         residue=outputs['residue'],
         biomass_with_resid=outputs['biomass_with_resid'],
         net_rate_with_resid=outputs['net_rate_with_resid'],
         lysis_rate_with_resid=outputs['lysis_rate_with_resid'],
         scenario_params=scenario_params)